# Notebook 04: Epidemiological Analytics

## Overview
This notebook calculates advanced epidemiological metrics, trends, and anomaly detection for cholera surveillance.

## Prerequisites
- Notebook 03 completed successfully
- Gold tables: `dim_country`, `dim_date`, `fact_cholera_cases`, `fact_cholera_deaths`

## Inputs
- Gold layer dimensional model

## Outputs
- `gold.epi_analytics_weekly` - Weekly epidemiological summary with trends
- `gold.epi_country_trends` - Country-level time series analysis
- `gold.epi_hotspot_detection` - Anomaly detection results

## Execution Time
~1-2 minutes

In [ ]:
# ============================================
# ENVIRONMENT DETECTION & CONFIGURATION
# ============================================

import os
import sys
from pathlib import Path

IS_FABRIC = os.path.exists('/lakehouse/default')

if IS_FABRIC:
    print("🌐 Running in Microsoft Fabric")
    GOLD_TABLE_PATH = "/lakehouse/default/Tables/gold"
else:
    print("💻 Running locally")
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(project_root / 'src'))
    GOLD_TABLE_PATH = str(project_root / "data" / "gold_tables")

print(f"Gold Path: {GOLD_TABLE_PATH}")

In [ ]:
# ============================================
# IMPORTS
# ============================================

import pandas as pd
import numpy as np
from datetime import datetime
from typing import Dict, List
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Imports successful")

In [ ]:
# ============================================
# LOAD GOLD DATA
# ============================================

print("\n📂 Loading Gold layer data...\n")

try:
    if IS_FABRIC:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        
        df_dim_country = spark.table("gold.dim_country").toPandas()
        df_dim_date = spark.table("gold.dim_date").toPandas()
        df_fact_cases = spark.table("gold.fact_cholera_cases").toPandas()
        df_fact_deaths = spark.table("gold.fact_cholera_deaths").toPandas()
    else:
        df_dim_country = pd.read_parquet(Path(GOLD_TABLE_PATH) / "dim_country.parquet")
        df_dim_date = pd.read_parquet(Path(GOLD_TABLE_PATH) / "dim_date.parquet")
        df_fact_cases = pd.read_parquet(Path(GOLD_TABLE_PATH) / "fact_cholera_cases.parquet")
        df_fact_deaths = pd.read_parquet(Path(GOLD_TABLE_PATH) / "fact_cholera_deaths.parquet")
    
    print(f"✅ Loaded {len(df_dim_country)} countries")
    print(f"✅ Loaded {len(df_dim_date)} dates")
    print(f"✅ Loaded {len(df_fact_cases)} case records")
    print(f"✅ Loaded {len(df_fact_deaths)} death records")
    
except Exception as e:
    logger.error(f"Error loading Gold data: {e}")
    raise

In [ ]:
# ============================================
# PREPARE ANALYTICS DATASET
# ============================================

print("\n🔄 Preparing analytics dataset...\n")

# Join facts with dimensions
df_analytics = df_fact_cases.merge(
    df_fact_deaths[['report_key', 'country_key', 'new_deaths', 'cfr_percent']],
    on=['report_key', 'country_key'],
    how='left'
)

df_analytics = df_analytics.merge(
    df_dim_country[['country_key', 'country_code', 'country_name', 'au_region', 'population']],
    on='country_key',
    how='left'
)

df_analytics = df_analytics.merge(
    df_dim_date[['date_key', 'date', 'epi_year', 'epi_week']],
    on='date_key',
    how='left'
)

# Convert date to datetime
if df_analytics['date'].dtype == 'object':
    df_analytics['date'] = pd.to_datetime(df_analytics['date'])

# Sort by country and date
df_analytics = df_analytics.sort_values(['country_code', 'date'])

print(f"✅ Analytics dataset prepared: {len(df_analytics)} records")
print(f"   - Countries: {df_analytics['country_code'].nunique()}")
print(f"   - Weeks: {df_analytics['epi_week'].nunique()}")

In [ ]:
# ============================================
# CALCULATE EPIDEMIOLOGICAL METRICS
# ============================================

print("\n📊 Calculating epidemiological metrics...\n")

# 1. Week-over-week growth rate
df_analytics['cases_prev_week'] = df_analytics.groupby('country_code')['new_cases'].shift(1)
df_analytics['growth_rate_pct'] = (
    (df_analytics['new_cases'] - df_analytics['cases_prev_week']) / 
    df_analytics['cases_prev_week'].replace(0, np.nan) * 100
).round(2)

# 2. 4-week moving average (cases)
df_analytics['cases_4wk_ma'] = (
    df_analytics.groupby('country_code')['new_cases']
    .rolling(window=4, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
).round(2)

# 3. 4-week moving average (deaths)
df_analytics['deaths_4wk_ma'] = (
    df_analytics.groupby('country_code')['new_deaths']
    .rolling(window=4, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
).round(2)

# 4. Cumulative metrics by country
df_analytics['cumulative_cases_country'] = df_analytics.groupby('country_code')['new_cases'].cumsum()
df_analytics['cumulative_deaths_country'] = df_analytics.groupby('country_code')['new_deaths'].cumsum()

# 5. Attack rate (cumulative cases per 100k population)
df_analytics['attack_rate_cumulative'] = (
    df_analytics['cumulative_cases_country'] / df_analytics['population'] * 100000
).round(2)

print("✅ Calculated growth rates and moving averages")
print(f"   - Average growth rate: {df_analytics['growth_rate_pct'].mean():.2f}%")
print(f"   - Max 4-week MA (cases): {df_analytics['cases_4wk_ma'].max():.0f}")

In [ ]:
# ============================================
# ANOMALY DETECTION (Modified Z-Score)
# ============================================

print("\n🚨 Running anomaly detection...\n")

def calculate_modified_zscore(series):
    """
    Calculate Modified Z-Score using Median Absolute Deviation (MAD).
    More robust to outliers than standard Z-score.
    """
    median = series.median()
    mad = np.median(np.abs(series - median))
    
    if mad == 0:
        return pd.Series([0] * len(series), index=series.index)
    
    modified_z = 0.6745 * (series - median) / mad
    return modified_z

# Calculate modified Z-score for cases by country
df_analytics['cases_modified_z'] = (
    df_analytics.groupby('country_code')['new_cases']
    .transform(calculate_modified_zscore)
).round(2)

# Flag anomalies (threshold = 2.5)
ANOMALY_THRESHOLD = 2.5
df_analytics['is_anomaly'] = np.abs(df_analytics['cases_modified_z']) > ANOMALY_THRESHOLD
df_analytics['anomaly_severity'] = pd.cut(
    np.abs(df_analytics['cases_modified_z']),
    bins=[0, 2.5, 3.5, np.inf],
    labels=['NORMAL', 'WARNING', 'CRITICAL']
)

anomaly_count = df_analytics['is_anomaly'].sum()
print(f"✅ Anomaly detection complete")
print(f"   - Anomalies detected: {anomaly_count}")
print(f"   - WARNING: {(df_analytics['anomaly_severity'] == 'WARNING').sum()}")
print(f"   - CRITICAL: {(df_analytics['anomaly_severity'] == 'CRITICAL').sum()}")

if anomaly_count > 0:
    print("\n🚨 Anomalies Detected:")
    anomalies = df_analytics[df_analytics['is_anomaly']][[
        'country_name', 'epi_year', 'epi_week', 'new_cases', 
        'cases_4wk_ma', 'cases_modified_z', 'anomaly_severity'
    ]]
    display(anomalies)

In [ ]:
# ============================================
# CREATE WEEKLY SUMMARY
# ============================================

print("\n📊 Creating weekly epidemiological summary...\n")

# Aggregate by week
df_weekly_summary = df_analytics.groupby(['epi_year', 'epi_week', 'date']).agg({
    'new_cases': 'sum',
    'new_deaths': 'sum',
    'confirmed_cases': 'sum',
    'suspected_cases': 'sum',
    'country_code': 'nunique',
    'incidence_rate': 'mean',
    'cfr_percent': 'mean',
    'is_anomaly': 'sum'
}).reset_index()

# Rename columns
df_weekly_summary = df_weekly_summary.rename(columns={
    'country_code': 'affected_countries',
    'is_anomaly': 'anomaly_count'
})

# Calculate weekly CFR
df_weekly_summary['weekly_cfr'] = (
    df_weekly_summary['new_deaths'] / df_weekly_summary['new_cases'] * 100
).round(2)

# Calculate week-over-week change
df_weekly_summary['cases_prev_week'] = df_weekly_summary['new_cases'].shift(1)
df_weekly_summary['weekly_change_pct'] = (
    (df_weekly_summary['new_cases'] - df_weekly_summary['cases_prev_week']) / 
    df_weekly_summary['cases_prev_week'].replace(0, np.nan) * 100
).round(2)

# Add cumulative totals
df_weekly_summary['cumulative_cases'] = df_weekly_summary['new_cases'].cumsum()
df_weekly_summary['cumulative_deaths'] = df_weekly_summary['new_deaths'].cumsum()

# Add audit column
df_weekly_summary['created_at'] = datetime.now()

print(f"✅ Created weekly summary: {len(df_weekly_summary)} weeks")
print("\n📋 Sample Weekly Summary:")
display(df_weekly_summary[[
    'epi_year', 'epi_week', 'new_cases', 'new_deaths', 
    'weekly_cfr', 'affected_countries', 'anomaly_count'
]].head())

In [ ]:
# ============================================
# CREATE COUNTRY TRENDS
# ============================================

print("\n📈 Creating country-level trends...\n")

df_country_trends = df_analytics[[
    'country_key', 'country_code', 'country_name', 'au_region',
    'date', 'epi_year', 'epi_week',
    'new_cases', 'new_deaths', 'cumulative_cases_country', 'cumulative_deaths_country',
    'cases_4wk_ma', 'deaths_4wk_ma', 'growth_rate_pct',
    'incidence_rate', 'attack_rate_cumulative', 'cfr_percent'
]].copy()

# Add trend direction
df_country_trends['trend_direction'] = df_country_trends['growth_rate_pct'].apply(
    lambda x: 'INCREASING' if x > 5 else ('DECREASING' if x < -5 else 'STABLE')
)

# Add audit column
df_country_trends['created_at'] = datetime.now()

print(f"✅ Created country trends: {len(df_country_trends)} records")
print(f"   - Countries: {df_country_trends['country_code'].nunique()}")
print(f"   - Trend distribution:")
print(df_country_trends['trend_direction'].value_counts())

In [ ]:
# ============================================
# CREATE HOTSPOT DETECTION
# ============================================

print("\n🔥 Creating hotspot detection...\n")

# Filter to anomalies only
df_hotspots = df_analytics[df_analytics['is_anomaly']].copy()

if len(df_hotspots) > 0:
    df_hotspots = df_hotspots[[
        'country_key', 'country_code', 'country_name', 'au_region',
        'date', 'epi_year', 'epi_week',
        'new_cases', 'cases_4wk_ma', 'cases_modified_z',
        'anomaly_severity', 'incidence_rate'
    ]]
    
    # Add detection timestamp
    df_hotspots['detection_date'] = datetime.now().date()
    df_hotspots['created_at'] = datetime.now()
    
    print(f"✅ Created hotspot detection: {len(df_hotspots)} hotspots")
    print("\n🔥 Hotspot Summary:")
    display(df_hotspots[[
        'country_name', 'epi_year', 'epi_week', 'new_cases', 
        'cases_modified_z', 'anomaly_severity'
    ]])
else:
    # Create empty DataFrame with schema
    df_hotspots = pd.DataFrame(columns=[
        'country_key', 'country_code', 'country_name', 'au_region',
        'date', 'epi_year', 'epi_week',
        'new_cases', 'cases_4wk_ma', 'cases_modified_z',
        'anomaly_severity', 'incidence_rate',
        'detection_date', 'created_at'
    ])
    print("✅ No hotspots detected (all values within normal range)")

In [ ]:
# ============================================
# SAVE TO GOLD LAYER
# ============================================

print("\n💾 Saving analytics to Gold layer...\n")

try:
    if IS_FABRIC:
        spark_weekly = spark.createDataFrame(df_weekly_summary)
        spark_trends = spark.createDataFrame(df_country_trends)
        spark_hotspots = spark.createDataFrame(df_hotspots)
        
        spark_weekly.write.format("delta").mode("overwrite").saveAsTable("gold.epi_analytics_weekly")
        spark_trends.write.format("delta").mode("overwrite").saveAsTable("gold.epi_country_trends")
        spark_hotspots.write.format("delta").mode("overwrite").saveAsTable("gold.epi_hotspot_detection")
        
        print("✅ Delta tables created in Fabric Lakehouse")
    else:
        # Convert datetime columns to strings
        for df, name in [
            (df_weekly_summary, 'epi_analytics_weekly'),
            (df_country_trends, 'epi_country_trends'),
            (df_hotspots, 'epi_hotspot_detection')
        ]:
            df_save = df.copy()
            for col in df_save.columns:
                if df_save[col].dtype == 'datetime64[ns]' or 'datetime' in str(df_save[col].dtype):
                    df_save[col] = df_save[col].astype(str)
            
            df_save.to_parquet(
                Path(GOLD_TABLE_PATH) / f"{name}.parquet",
                index=False,
                engine='pyarrow'
            )
            print(f"✅ Saved {name}.parquet ({len(df_save)} rows)")
        
        print(f"\n✅ All files saved to: {GOLD_TABLE_PATH}")
        
except Exception as e:
    logger.error(f"Error saving analytics: {e}")
    raise

print("\n✅ Epidemiological analytics complete!")

## Validation & Testing

In [ ]:
# ============================================
# VALIDATION & TESTING
# ============================================

print("\n🔍 Running validation checks...\n")

# Test 1: Analytics tables created
assert len(df_weekly_summary) > 0, "Weekly summary is empty"
assert len(df_country_trends) > 0, "Country trends is empty"
print(f"✅ Analytics tables populated")

# Test 2: Metrics calculated
assert df_weekly_summary['weekly_cfr'].notna().sum() > 0, "CFR not calculated"
assert df_country_trends['cases_4wk_ma'].notna().sum() > 0, "Moving averages not calculated"
print(f"✅ All metrics calculated")

# Test 3: Anomaly detection ran
assert 'anomaly_severity' in df_analytics.columns, "Anomaly detection not run"
print(f"✅ Anomaly detection completed")

# Test 4: Data quality
print("\n📊 Analytics Summary:")
print(f"  - Weekly records: {len(df_weekly_summary)}")
print(f"  - Country trend records: {len(df_country_trends)}")
print(f"  - Hotspots detected: {len(df_hotspots)}")
print(f"  - Total cases analyzed: {df_weekly_summary['new_cases'].sum():,}")
print(f"  - Total deaths analyzed: {df_weekly_summary['new_deaths'].sum():,}")
print(f"  - Overall CFR: {(df_weekly_summary['new_deaths'].sum() / df_weekly_summary['new_cases'].sum() * 100):.2f}%")

print("\n✅ All validation checks passed!")

## Next Steps

1. **Review analytics results** above
2. **Investigate any hotspots** detected
3. **Proceed to Notebook 05** for ML forecasting

## Outputs Created

- `gold.epi_analytics_weekly` - Weekly epidemiological summary with trends
- `gold.epi_country_trends` - Country-level time series with moving averages
- `gold.epi_hotspot_detection` - Anomaly detection results

**Ready for forecasting!** ✅